# Water / steam property test

Functional test for the IAPWS-IF97 water/steam property provider.

The notebook is intentionally user-facing:
1. verify that the required API is importable and returns sensible values,
2. evaluate water/steam properties for a practical matrix of `T + p` points,
3. display a table with the complete property set used by KalKalori.

Core units:
- `T`: K
- `p`: Pa
- `rho`: kg/m³
- `mu`: Pa·s
- `k`: W/(m·K)
- `cp`: J/(kg·K)
- `h`: J/kg

In [ ]:
from pathlib import Path
import sys
import math

import pandas as pd
from iapws import IAPWS97

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from core.properties import (
    IAPWS97WaterSteamProvider,
    water_steam_props_iapws97,
)

from core.properties.adapters import (
    to_internal_fluid_props,
    to_internal_pressure_drop_fluid_props,
    to_outside_fluid_props,
)

from core.properties.averaging import (
    mean_temperature,
    mean_transport_props,
)

# Basic smoke test: ordinary T+p point.
water_ref = water_steam_props_iapws97(
    T=293.15,
    p=101325.0,
)

assert 995.0 < water_ref.transport.rho < 1000.0
assert 0.0009 < water_ref.transport.mu < 0.0011
assert 0.58 < water_ref.transport.k < 0.62
assert 4100.0 < water_ref.transport.cp < 4300.0
assert 80_000.0 < water_ref.h < 90_000.0

# Smoke test: provider shape used by property layer.
provider = IAPWS97WaterSteamProvider()
provider_props = provider.at(T=293.15, p=101325.0)

assert provider_props.rho == water_ref.transport.rho
assert provider_props.mu == water_ref.transport.mu
assert provider_props.k == water_ref.transport.k
assert provider_props.cp == water_ref.transport.cp

# Smoke test: saturated states by p+x and T+x.
sat_liq_p = water_steam_props_iapws97(p=101325.0, x=0.0)
sat_vap_p = water_steam_props_iapws97(p=101325.0, x=1.0)

sat_liq_t = water_steam_props_iapws97(T=423.15, x=0.0)
sat_vap_t = water_steam_props_iapws97(T=423.15, x=1.0)

assert sat_liq_p.transport.rho > sat_vap_p.transport.rho
assert sat_liq_p.h < sat_vap_p.h
assert sat_liq_t.transport.rho > sat_vap_t.transport.rho
assert sat_liq_t.h < sat_vap_t.h

print("Water/steam imports and smoke test passed.")

In [ ]:
# Practical matrix: water/steam properties for engineering points.
#
# Points are deliberately chosen away from exact saturation boundaries,
# because this test is about stable point-property retrieval, not two-phase solving.

water_points = [
    {"case": "ambient liquid water", "T_C": 20.0, "p_bar": 1.01325},
    {"case": "warm liquid water", "T_C": 60.0, "p_bar": 1.01325},
    {"case": "hot liquid water", "T_C": 90.0, "p_bar": 1.01325},
    {"case": "pressurized liquid", "T_C": 150.0, "p_bar": 5.0},
    {"case": "low-pressure superheated steam", "T_C": 120.0, "p_bar": 1.0},
    {"case": "superheated steam", "T_C": 200.0, "p_bar": 10.0},
    {"case": "saturated liquid at 1 atm", "p_bar": 1.01325, "x": 0.0},
    {"case": "saturated steam at 1 atm", "p_bar": 1.01325, "x": 1.0},
    {"case": "saturated liquid at 16 atm", "p_bar": 16.0, "x": 0.0},
    {"case": "saturated steam at 16 atm", "p_bar": 16.0, "x": 1.0},
    {"case": "saturated liquid at 150C", "T_C": 150.0, "x": 0.0},
    {"case": "saturated steam at 150C", "T_C": 150.0, "x": 1.0},
    {"case": "low-pressure superheated steam", "T_C": 120.0, "p_bar": 1.01325},
    {"case": "superheated steam", "T_C": 200.0, "p_bar": 5.0},
]

rows = []

for point in water_points:
    warning_codes = []

    try:
        has_T = "T_C" in point
        has_p = "p_bar" in point
        has_x = "x" in point

        if has_T and has_p and not has_x:
            input_mode = "T/p"

            T = point["T_C"] + 273.15
            p = point["p_bar"] * 1.0e5

            result = water_steam_props_iapws97(
                T=T,
                p=p,
            )

            T_C_display = point["T_C"]
            p_bar_display = point["p_bar"]
            x_display = math.nan

        elif has_p and has_x and not has_T:
            input_mode = "p/x"

            p = point["p_bar"] * 1.0e5
            x = point["x"]

            result = water_steam_props_iapws97(
                p=p,
                x=x,
            )

            # Display-only saturation temperature.
            sat_state = IAPWS97(P=p / 1.0e6, x=x)

            T_C_display = sat_state.T - 273.15
            p_bar_display = point["p_bar"]
            x_display = x

        elif has_T and has_x and not has_p:
            input_mode = "T/x"

            T = point["T_C"] + 273.15
            x = point["x"]

            result = water_steam_props_iapws97(
                T=T,
                x=x,
            )

            # Display-only saturation pressure.
            sat_state = IAPWS97(T=T, x=x)

            T_C_display = point["T_C"]
            p_bar_display = sat_state.P * 10.0  # MPa -> bar
            x_display = x

        else:
            raise ValueError(
                "Point must contain exactly one supported input mode: "
                "T_C+p_bar, p_bar+x, or T_C+x."
            )

        for warning in result.warnings:
            warning_codes.append(warning.code)

        rows.append(
            {
                "case": point["case"],
                "input_mode": input_mode,
                "T_C": T_C_display,
                "p_bar": p_bar_display,
                "x": x_display,
                "phase": result.phase,
                "rho_kg_m3": result.transport.rho,
                "mu_Pa_s": result.transport.mu,
                "k_W_mK": result.transport.k,
                "cp_J_kgK": result.transport.cp,
                "h_kJ_kg": result.h / 1000.0,
                "warnings": ", ".join(sorted(set(warning_codes))),
                "error": "",
            }
        )

    except Exception as exc:
        rows.append(
            {
                "case": point["case"],
                "input_mode": "ERROR",
                "T_C": point.get("T_C", math.nan),
                "p_bar": point.get("p_bar", math.nan),
                "x": point.get("x", math.nan),
                "phase": "",
                "rho_kg_m3": math.nan,
                "mu_Pa_s": math.nan,
                "k_W_mK": math.nan,
                "cp_J_kgK": math.nan,
                "h_kJ_kg": math.nan,
                "warnings": "",
                "error": str(exc),
            }
        )

water_df = pd.DataFrame(rows)

In [ ]:
# Practical checks.
ambient = water_df.loc[water_df["case"] == "ambient liquid water"].iloc[0]
sat_liq_p = water_df.loc[water_df["case"] == "saturated liquid at 1 atm"].iloc[0]
sat_vap_p = water_df.loc[water_df["case"] == "saturated steam at 1 atm"].iloc[0]
sat_liq_t = water_df.loc[water_df["case"] == "saturated liquid at 150C"].iloc[0]
sat_vap_t = water_df.loc[water_df["case"] == "saturated steam at 150C"].iloc[0]

assert ambient["error"] == ""
assert 995.0 < ambient["rho_kg_m3"] < 1000.0
assert 0.0009 < ambient["mu_Pa_s"] < 0.0011
assert 0.58 < ambient["k_W_mK"] < 0.62
assert 4100.0 < ambient["cp_J_kgK"] < 4300.0

assert sat_liq_p["error"] == ""
assert sat_vap_p["error"] == ""
assert 99.0 < sat_liq_p["T_C"] < 101.0
assert 99.0 < sat_vap_p["T_C"] < 101.0
assert sat_liq_p["rho_kg_m3"] > sat_vap_p["rho_kg_m3"]
assert sat_liq_p["h_kJ_kg"] < sat_vap_p["h_kJ_kg"]

assert sat_liq_t["error"] == ""
assert sat_vap_t["error"] == ""
assert 4.5 < sat_liq_t["p_bar"] < 5.0
assert 4.5 < sat_vap_t["p_bar"] < 5.0
assert sat_liq_t["rho_kg_m3"] > sat_vap_t["rho_kg_m3"]
assert sat_liq_t["h_kJ_kg"] < sat_vap_t["h_kJ_kg"]

print("Water/steam functional matrix test passed.")

water_df